#### Phase 2 Part 1: Authentication

In [2]:
import os
import os.path
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

In [ ]:
# Define the permissions needed (Read & Modify Gmail labels)
SCOPES = ['https://www.googleapis.com/auth/gmail.modify']

def authenticate_gmail(account):
    token_file = f'token_{account}.json'
    creds = None
    if os.path.exists(token_file):
        creds = Credentials.from_authorized_user_file(token_file, SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            try:
                creds.refresh(Request())
            except Exception as e:
                print(f"Refresh failed ({e}), falling back to full login...")
                flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
                creds = flow.run_local_server(port=0)
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0);

        with open(token_file, 'w') as token:
            token.write(creds.to_json())

    service = build('gmail', 'v1', credentials=creds)
    return service

gmail_service = authenticate_gmail('personal')

print("Successfully connected to Gmail API!")

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=658547031447-cjrlo1b2hp76c57aupb1lmfiau6j6gre.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A55743%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.modify&state=QCnLBYKjvcXh6iGur2tns6TCldG3VH&code_challenge=jaZ38rYIGx3mVIIQSxscrpaAT1FZv0op7-ZLMvEe6kE&code_challenge_method=S256&access_type=offline


#### Phase 2 Part 2: Fetching Unread Emails


In [8]:
import html
import base64
import re
from bs4 import BeautifulSoup  # pip install beautifulsoup4, if not already installed

def extract_email_body(payload):
    """
    Recursively walks the Gmail message payload to find and decode the body text.
    Prefers text/plain; falls back to stripping HTML from text/html if that's all
    that's available. Returns a plain-text string.
    """
    body_data = None

    def find_part(part):
        nonlocal body_data
        mime_type = part.get('mimeType', '')
        data = part.get('body', {}).get('data')

        if mime_type == 'text/plain' and data:
            body_data = ('plain', data)
            return True  # prefer plain, stop searching once found
        if mime_type == 'text/html' and data and body_data is None:
            body_data = ('html', data)  # keep looking in case plain shows up later

        for sub_part in part.get('parts', []):
            if find_part(sub_part):
                return True
        return False

    find_part(payload)

    if body_data is None:
        return ""

    kind, data = body_data
    decoded = base64.urlsafe_b64decode(data).decode('utf-8', errors='replace')

    if kind == 'html':
        decoded = BeautifulSoup(decoded, 'html.parser').get_text(separator=' ', strip=True)

    return decoded.strip()

def get_unread_emails(service, max_results=3):
    results = service.users().messages().list(
        userId='me', q='is:unread in:inbox', maxResults=max_results
    ).execute()

    messages = results.get('messages', [])
    if not messages:
        print("No unread messages in your inbox!")
        return []

    email_list = []
    for msg in messages:
        full_msg = service.users().messages().get(
            userId='me', id=msg['id'], format='full'
        ).execute()

        headers = full_msg['payload']['headers']
        sender = html.unescape(next((h['value'] for h in headers if h['name'] == 'From'), 'Unknown'))
        subject = html.unescape(next((h['value'] for h in headers if h['name'] == 'Subject'), 'No Subject'))
        snippet = html.unescape(full_msg.get('snippet', ''))
        body = extract_email_body(full_msg['payload'])  # NEW

        email_info = {
            'id': msg['id'],
            'sender': sender,
            'subject': subject,
            'snippet': snippet,
            'body': body,  # NEW
        }
        email_list.append(email_info)

    return email_list

unread_emails = get_unread_emails(gmail_service)

#### Phase 2 Part 3: Modifying Gmail Labels

In [9]:
import json
import os

PROCESSED_IDS_FILE = "processed_ids.json"

def load_processed_ids():
    if os.path.exists(PROCESSED_IDS_FILE):
        with open(PROCESSED_IDS_FILE, "r") as f:
            return set(json.load(f))
    return set()

def save_processed_ids(ids):
    with open(PROCESSED_IDS_FILE, "w") as f:
        json.dump(list(ids), f)        

def get_or_create_label(service, label_name):
    # Checks if a label exists in Gmail. If it doesn't exist, create a label

    # Fetch all existing user labels
    results = service.users().labels().list(userId='me').execute()
    labels = results.get('labels',[])

    # Check if the label exists
    for label in labels:
        if label['name'].lower() == label_name.lower():
            return label['id']

    # Create the label if it doesn't already exist
    label_object = {
        'name': label_name,
        'labelListVisibility': 'labelShow',
        'messageListVisibility': 'show'
    }

    created_label = service.users().labels().create(
        userId = 'me',
        body = label_object
    ).execute()

    print(f"Created new Gmail label: '{created_label}'")
    return created_label['id']

def apply_label(service, msg_id, label_id):
    # Applies label without touching read status

    body = {'addLabelIds': [label_id]}
    updated_message = service.users().messages().modify(
        userId='me', id=msg_id,body=body
    ).execute()
    print(f"Labeled message {msg_id} as important (left unread status untouched).")
    return updated_message


#### Phase 3 Part 1: Gemini Email Classification

In [10]:
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()
client = genai.Client()

# Email Importance Schema, or the Output Shape
class EmailImportance(BaseModel):
    is_important: bool = Field(
        description="True if this email is important and requires Exel's personal attention; "
                    "False for newsletters, promotions, spam, or general club-wide blasts "
                    "that don't need direct action."
    )
    reason: str = Field(
        description="One short sentence explaining why it was flagged important or not."
    )

# Email Classification function
def analyze_email_with_ai(sender, subject, body):
    content = body[:3000] if body else "(no body content available)"

    prompt = f"""
    Determine whether this email is IMPORTANT and requires Exel Uy's direct, personal attention.
    Note: If this is a forwarded email, evaluate the core content and context of the
    original message inside the body, not just the forwarding note on top.

    Mark as important if it:
    - Is addressed to Exel specifically or requires a personal decision/reply from him
      (can be a general inquiry from a club member regarding Culture Show applications,
      auditions, or logistics)
    - Involves a deadline, task, or responsibility tied to his role as Culture Show Director
    - Comes from a known collaborator (e.g. Sam Cubbage, the Production Coordinator and theater venue contact) —
      any email from a named collaborator should default to important
    - Comes from club officers, university administration, or sponsors,
      regarding show logistics, approvals, or a request that needs a response

    Mark as NOT important if it's a promotional email, newsletter, automated marketing,
    or a general announcement that doesn't need individual action.

    Sender: {sender}
    Subject: {subject}
    Body: {content}
    """
    try:
        response = client.models.generate_content(
            model='gemini-3.6-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=EmailImportance,
                system_instruction="You are a precise email triage assistant for Exel Uy, "
                                    "Culture Show Director, screening his inbox for what "
                                    "genuinely needs his attention."
            ),
        )
        return EmailImportance.model_validate_json(response.text)
    except Exception as e:
        print(f"Skipped classification for '{subject}' due to error: {e}")
        return None

#### Phase 4: Main Agent Loop

In [11]:
import time

def process_inbox_agent(service, max_emails=5):
    TARGET_LABEL_NAME = "2026-27 Exel Uy"
    label_id = get_or_create_label(service, TARGET_LABEL_NAME)  # fetched ONCE, not per email

    processed_ids = load_processed_ids()
    emails = get_unread_emails(service, max_results=max_emails)

    if not emails:
        print("No unread emails to process.")
        return

    new_emails = [e for e in emails if e['id'] not in processed_ids]
    print(f"{len(emails)} unread, {len(new_emails)} not yet classified.\n")

    for email in new_emails:
        analysis = analyze_email_with_ai(
            sender=email['sender'], subject=email['subject'], body=email['body']
        )

        if analysis is None:
            continue  # NEW — skip this email, don't mark as processed, retry next run

        if analysis.is_important:
            apply_label(service, email['id'], label_id)
            print(f"IMPORTANT: '{email['subject']}' — {analysis.reason}")
        else:
            print(f"Skipped (not important): '{email['subject']}' — {analysis.reason}")

        processed_ids.add(email['id'])

        time.sleep(15) # double check if this is a valid approach
    save_processed_ids(processed_ids)

process_inbox_agent(gmail_service, max_emails=3)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


3 unread, 3 not yet classified.

Skipped classification for 'Security alert' due to error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Skipped (not important): 'This week: 49ers host the Dolphins' — This is a promotional marketing email for event tickets and does not require direct action from Exel.
Skipped classification for '[GitHub] A first-party GitHub OAuth application has been added to your account' due to error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
